# Week 5, Lab 3 — Pydantic AI on a local model


In [ ]:
WEEK = 'Week 5'
LAB = 'Lab 3 — Pydantic AI'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn pyautogen pydantic-ai openai
else:
    %pip install -q pyautogen pydantic-ai ollama openai


In [ ]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent

cfg = openai_client_kwargs()

class CourseFact(BaseModel):
    topic: str
    summary: str = Field(description="one sentence")
    difficulty: int = Field(ge=1, le=5)

try:
    from pydantic_ai.models.openai import OpenAIChatModel
    from pydantic_ai.providers.openai import OpenAIProvider
    model = OpenAIChatModel(cfg["model"], provider=OpenAIProvider(base_url=cfg["base_url"], api_key=cfg["api_key"]))
except Exception:
    from pydantic_ai.models.openai import OpenAIModel
    model = OpenAIModel(cfg["model"], base_url=cfg["base_url"], api_key=cfg["api_key"])

agent = Agent(model, output_type=CourseFact, instructions="Extract a CourseFact. Be literal.")
result = agent.run_sync("LangGraph lets you build stateful multi-actor LLM workflows as graphs.")
print(result.output)


Typed outputs are the point of this framework.
